# Challenge Sprint IV — GoodWe
**Modelagem Matemática e Computacional**

Verificação computacional (Python) da resolução analítica. Segue o padrão das sprints anteriores: resolução passo a passo no papel e conferência com NumPy / SymPy.

**Conexão com as sprints anteriores**
- Sprint I (derivadas): potência instantânea P(t), em kW.
- Sprint II (máximos e mínimos): preço/tarifa e lucro L(p); extremos no ponto crítico ou na borda do intervalo.
- Sprint 3 (integrais): energia E = ∫P dt, em kWh.
- Sprint IV (vetores e sistemas): energia por período (kWh), custo em R$ e distribuição das potências (kW).

Cadeia de unidades: kW → (× tempo) → kWh → (× R$/kWh) → R$.

## Parte 1 — Energia solar e vetores
Os valores numéricos do enunciado chegaram ilegíveis. Abaixo, a solução é **simbólica** (vale para quaisquer valores). Na célula de entrada, troque os números de EXEMPLO pelos do enunciado.

In [14]:
import sympy as sp
import numpy as np

p = sp.symbols('p1:6', positive=True)
e = sp.symbols('e1:6', positive=True)
s = sp.symbols('s1:6', positive=True)

# a)
r = [sp.Max(0, e[i] - s[i]) for i in range(5)]
print("a) r =", r)

# b)
C_total = sum(p[i] * r[i] for i in range(5))
print("b) C =", C_total, "  [R$]")

a) r = [Max(0, e1 - s1), Max(0, e2 - s2), Max(0, e3 - s3), Max(0, e4 - s4), Max(0, e5 - s5)]
b) C = p1*Max(0, e1 - s1) + p2*Max(0, e2 - s2) + p3*Max(0, e3 - s3) + p4*Max(0, e4 - s4) + p5*Max(0, e5 - s5)   [R$]


In [15]:
p_num = np.array([0.80, 0.95, 1.10, 1.05, 0.90])
e_num = np.array([40, 60, 80, 60, 40])
s_num = np.array([10, 50, 90, 55, 5])

r_num = np.maximum(0, e_num - s_num)
parcelas = p_num * r_num
custo = np.dot(p_num, r_num)

print("r =", r_num, "kWh")
print("parcelas p_i*r_i =", parcelas, "R$")
print(f"Custo total = R$ {custo:.2f}")
assert abs(custo - parcelas.sum()) < 1e-9

r = [30 10  0  5 35] kWh
parcelas p_i*r_i = [24.    9.5   0.    5.25 31.5 ] R$
Custo total = R$ 70.25


**c) Significado e unidade.** Cada parcela $p_i r_i$ é o preço do kWh no intervalo $i$ (R\$/kWh) multiplicado pela energia comprada da rede (kWh). Os kWh se cancelam e sobra R\$: cada parcela é o custo do intervalo e a soma é o custo diário da energia da rede. Onde a solar cobre a demanda ($s_i > e_i$), $r_i = 0$ e a parcela é nula.

## Parte 2 — Potência dos carregadores e sistema linear (12h–14h)
Variáveis: $x$ = veículos a 7 kW, $y$ = a 11 kW, $z$ = a 22 kW.

1. $x + y + z = 6$ (total de veículos)
2. $7x + 11y + 22z = 69$ (potência total, kW)
3. $y + z = 4$ (veículos em 11 kW ou 22 kW)

In [18]:
A = np.array([[1, 1, 1],
              [7, 11, 22],
              [0, 1, 1]], dtype=float)
b = np.array([6, 69, 4], dtype=float)

# Eliminação gaussiana passo a passo
Ab = np.column_stack([A, b])
print("Matriz aumentada inicial:\n", Ab)

Ab[1] = Ab[1] - 7 * Ab[0]
print("\nL2 <- L2 - 7*L1:\n", Ab)

Ab[[1, 2]] = Ab[[2, 1]]
print("\nTroca L2 <-> L3:\n", Ab)

Ab[2] = Ab[2] - 4 * Ab[1]
print("\nL3 <- L3 - 4*L2 (escalonada):\n", Ab)

# Substituição regressiva
z = Ab[2, 3] / Ab[2, 2]
y = (Ab[1, 3] - Ab[1, 2] * z) / Ab[1, 1]
x = (Ab[0, 3] - Ab[0, 1] * y - Ab[0, 2] * z) / Ab[0, 0]
print(f"\nz = {z:.0f}, y = {y:.0f}, x = {x:.0f}")

Matriz aumentada inicial:
 [[ 1.  1.  1.  6.]
 [ 7. 11. 22. 69.]
 [ 0.  1.  1.  4.]]

L2 <- L2 - 7*L1:
 [[ 1.  1.  1.  6.]
 [ 0.  4. 15. 27.]
 [ 0.  1.  1.  4.]]

Troca L2 <-> L3:
 [[ 1.  1.  1.  6.]
 [ 0.  1.  1.  4.]
 [ 0.  4. 15. 27.]]

L3 <- L3 - 4*L2 (escalonada):
 [[ 1.  1.  1.  6.]
 [ 0.  1.  1.  4.]
 [ 0.  0. 11. 11.]]

z = 1, y = 3, x = 2


In [19]:
# Verificação com NumPy e SymPy
print("np.linalg.solve ->", np.linalg.solve(A, b))
X, Y, Z = sp.symbols('x y z')
print("sympy ->", sp.solve([X + Y + Z - 6, 7*X + 11*Y + 22*Z - 69, Y + Z - 4], [X, Y, Z]))
print("posto de A =", np.linalg.matrix_rank(A), "(3 pivôs: solução única)")
print("potência total =", 7*x + 11*y + 22*z, "kW")

np.linalg.solve -> [2. 3. 1.]
sympy -> {x: 2, y: 3, z: 1}
posto de A = 3 (3 pivôs: solução única)
potência total = 69.0 kW


**Resposta:** 2 veículos a 7 kW, 3 veículos a 11 kW e 1 veículo a 22 kW. O sistema tem solução única (três pivôs).

**Verificação cruzada com a Parte 1:** a potência de 69 kW por 2 h (12h–14h) corresponde a 138 kWh, que deve ser compatível com a demanda $e_3$ do enunciado.